# Interactive Brokers — Historical Data Pull

Connect to a locally running TWS (Trader Workstation) and pull historical bar data.
Saves directly to NautilusTrader's streaming catalog format (Arrow IPC / feather files)
so the data is immediately available in the Nautilus Automatron dashboard.

## Prerequisites
- TWS installed and running with API enabled (see setup instructions below)
- IB account with market data subscriptions

## TWS Installation & Setup

1. **Download TWS** from https://www.interactivebrokers.com/en/trading/download-tws.php
2. **Install** the `.dmg` file (Mac) and launch Trader Workstation
3. **Log in** with your IB username, password, and complete 2FA via the IBKR mobile app
4. **Enable API access:**
   - Go to **Edit > Global Configuration > API > Settings**
   - Check **Enable ActiveX and Socket Clients**
   - Set **Socket port** to `7496` (live) or `7497` (paper)
   - Uncheck **Read-Only API** if you want order execution (leave checked for data-only)
   - Click **Apply** and **OK**
5. **Keep TWS running** while using this notebook

In [ ]:
import datetime
import json
import socket
import uuid
from pathlib import Path

import pyarrow as pa
import pyarrow.ipc as ipc

from nautilus_trader.adapters.interactive_brokers.common import IBContract
from nautilus_trader.adapters.interactive_brokers.historical.client import (
    HistoricInteractiveBrokersClient,
)
from nautilus_trader.core.nautilus_pyo3.model import Bar as PyO3Bar
from nautilus_trader.serialization.arrow.serializer import ArrowSerializer

# === TWS Connection Settings ===
TWS_HOST = "127.0.0.1"
TWS_PORT = 7496  # 7496 = live, 7497 = paper
TWS_CLIENT_ID = 1

# === Catalog Settings ===
# This must match the NAUTILUS_STORE_PATH used by the Automatron server
CATALOG_PATH = Path("/Users/mordrax/code/nautilus_trader/backtest_catalog")


def check_tws_connection(host: str, port: int) -> bool:
    """Check if TWS is accepting connections on the given host:port."""
    try:
        with socket.create_connection((host, port), timeout=3):
            return True
    except (ConnectionRefusedError, TimeoutError, OSError):
        return False


def save_bars_to_catalog(
    bars: list,
    catalog_path: Path,
    run_id: str | None = None,
) -> str:
    """Save bars to the NautilusTrader streaming catalog format.

    Creates the directory structure:
        catalog_path/backtest/{run_id}/
            config.json
            bar/{bar_type}/{bar_type}_{ts_ns}.feather

    Returns the run_id used.
    """
    if not bars:
        raise ValueError("No bars to save")

    run_id = run_id or str(uuid.uuid4())
    run_dir = catalog_path / "backtest" / run_id
    bar_type_str = str(bars[0].bar_type)

    # Serialize bars to Arrow record batch (includes schema metadata)
    batch = ArrowSerializer.rust_defined_to_record_batch(bars, data_cls=PyO3Bar)

    # Create bar directory
    # Replace characters that aren't filesystem-safe
    safe_bar_type = bar_type_str.replace("/", "-").replace(":", "-")
    bar_dir = run_dir / "bar" / safe_bar_type
    bar_dir.mkdir(parents=True, exist_ok=True)

    # Write feather file
    ts_ns = bars[0].ts_event
    feather_path = bar_dir / f"{safe_bar_type}_{ts_ns}.feather"
    with open(feather_path, "wb") as f:
        writer = ipc.new_stream(f, batch.schema)
        writer.write_table(batch)
        writer.close()

    # Write minimal config.json
    config_path = run_dir / "config.json"
    if not config_path.exists():
        config = {
            "environment": "backtest",
            "trader_id": "BACKTESTER-001",
            "instance_id": None,
        }
        with open(config_path, "w") as f:
            json.dump(config, f, indent=2)

    return run_id


if check_tws_connection(TWS_HOST, TWS_PORT):
    print(f"TWS API detected at {TWS_HOST}:{TWS_PORT}")
else:
    print(f"WARNING: Cannot reach TWS at {TWS_HOST}:{TWS_PORT}")
    print()
    print("Please ensure:")
    print("  1. TWS (Trader Workstation) is running and you are logged in")
    print("  2. API is enabled: Edit > Global Configuration > API > Settings")
    print("     - 'Enable ActiveX and Socket Clients' is checked")
    print(f"     - Socket port is set to {TWS_PORT}")
    print("  3. If using paper trading, change TWS_PORT to 7497 above")
    print()
    print("Download TWS: https://www.interactivebrokers.com/en/trading/download-tws.php")

print(f"Catalog path: {CATALOG_PATH}")

## 1. Connect Historical Data Client

Connects to your locally running TWS for historical data requests.

> **Note:** If the connection fails, check the warning output from the cell above.

In [10]:
client = HistoricInteractiveBrokersClient(
    host=TWS_HOST,
    port=TWS_PORT,
    client_id=TWS_CLIENT_ID,
)

await client.connect()
print(f"Connected to TWS at {TWS_HOST}:{TWS_PORT}")

RuntimeError: Logging subsystem already initialized

## 2. Pull Historical Data

Configure the instrument, bar size, and date range below.
Two examples are provided: XAUUSD 1-minute and 5-minute bars.

### IB Rate Limits
- Max 60 requests per 10 minutes
- No identical requests within 15 seconds
- The client handles chunked requests automatically

### Available Bar Specifications
- `1-MINUTE-MIDPOINT`, `1-MINUTE-BID`, `1-MINUTE-LAST`
- `5-MINUTE-MIDPOINT`, `5-MINUTE-BID`, `5-MINUTE-LAST`
- `1-HOUR-MIDPOINT`, `1-HOUR-BID`, `1-HOUR-LAST`
- `1-DAY-MIDPOINT`, `1-DAY-BID`, `1-DAY-LAST`

> **Note:** For CFDs (like XAUUSD), use `MIDPOINT`. `BID`/`ASK` may not be available.

In [ ]:
# XAUUSD (Spot Gold) — available for non-US residents
# For US residents, use: IBContract(secType="FUT", symbol="GC", exchange="COMEX", lastTradeDateOrContractMonth="YYYYMM")
xauusd_contract = IBContract(
    secType="CFD",
    symbol="XAUUSD",
    exchange="SMART",
    currency="USD",
)

# Date range — start with 30 days to test, increase once confirmed working
# IB provides up to ~5 years of 1-minute data, but large requests are slow
end_date = datetime.datetime.now()
start_date = end_date - datetime.timedelta(days=30)

print(f"Contract: {xauusd_contract.symbol}")
print(f"Date range: {start_date.date()} to {end_date.date()}")

### Example 1: Pull 1-Minute Bars

In [ ]:
bars_1m = await client.request_bars(
    bar_specifications=["1-MINUTE-MIDPOINT"],
    start_date_time=start_date,
    end_date_time=end_date,
    tz_name="UTC",
    contracts=[xauusd_contract],
    use_rth=False,  # Include extended hours for forex/CFDs
    timeout=300,
)

print(f"Received {len(bars_1m)} 1-minute bars")
if bars_1m:
    print(f"First bar: {bars_1m[0]}")
    print(f"Last bar:  {bars_1m[-1]}")

In [ ]:
if bars_1m:
    run_id_1m = save_bars_to_catalog(bars_1m, CATALOG_PATH)
    print(f"Saved {len(bars_1m)} bars to catalog")
    print(f"Run ID: {run_id_1m}")
    print(f"Path: {CATALOG_PATH / 'backtest' / run_id_1m}")
else:
    print("No bars to save")

### Example 2: Pull 5-Minute Bars

In [ ]:
bars_5m = await client.request_bars(
    bar_specifications=["5-MINUTE-MIDPOINT"],
    start_date_time=start_date,
    end_date_time=end_date,
    tz_name="UTC",
    contracts=[xauusd_contract],
    use_rth=False,
    timeout=300,
)

print(f"Received {len(bars_5m)} 5-minute bars")
if bars_5m:
    print(f"First bar: {bars_5m[0]}")
    print(f"Last bar:  {bars_5m[-1]}")

In [ ]:
if bars_5m:
    run_id_5m = save_bars_to_catalog(bars_5m, CATALOG_PATH)
    print(f"Saved {len(bars_5m)} bars to catalog")
    print(f"Run ID: {run_id_5m}")
    print(f"Path: {CATALOG_PATH / 'backtest' / run_id_5m}")
else:
    print("No bars to save")

### Custom Pull

Modify the contract, bar spec, and date range below to pull any instrument.

In [ ]:
# === CONFIGURE YOUR PULL HERE ===
custom_contract = IBContract(
    secType="CFD",       # STK, FUT, OPT, CFD, CASH
    symbol="XAUUSD",     # IB symbol
    exchange="SMART",    # Exchange
    currency="USD",      # Currency
)

custom_bar_specs = ["1-MINUTE-MIDPOINT"]  # Change timeframe here
custom_start = datetime.datetime.now() - datetime.timedelta(days=30)
custom_end = datetime.datetime.now()
# ================================

custom_bars = await client.request_bars(
    bar_specifications=custom_bar_specs,
    start_date_time=custom_start,
    end_date_time=custom_end,
    tz_name="UTC",
    contracts=[custom_contract],
    use_rth=False,
    timeout=300,
)

print(f"Received {len(custom_bars)} bars")

if custom_bars:
    custom_run_id = save_bars_to_catalog(custom_bars, CATALOG_PATH)
    print(f"Saved to catalog, run ID: {custom_run_id}")
else:
    print("No bars to save")